# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ShamKottish/FlyRankML/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
%pip -q install duckdb huggingface_hub

import os
import getpass
import duckdb
import pandas as pd
import numpy as np

# Safely get Hugging Face token
HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

HF_TOKEN = HF_TOKEN or getpass.getpass(
    "Paste your Hugging Face READ token: "
)

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

FACT_MAR = (
    f"read_parquet('{REL}/"
    "fact_content_daily_performance/month=2026-03/*.parquet')"
)

FACT_APR = (
    f"read_parquet('{REL}/"
    "fact_content_daily_performance/month=2026-04/*.parquet')"
)

print("✓ Connected to FlyRank warehouse")
print("Feature window: March 2026")
print("Outcome window: April 2026")

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

I build one feature vector per pseudonymized client-content pair using only measurements observed during **March 2026**.

The feature vector includes search exposure, clicks, CTR, average position, the number of days with impressions, position variability, and a categorical position band. I also add an indicator for missing average position rather than silently treating missing position as zero.

The April 2026 measurements are kept separately and are used only to construct the later CTR-opportunity proxy. They are never included in the feature vector.

The final feature matrix contains only information that would have been available at the end of March.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ---------------------------------------------------------
# Build March features
# one row = one client + content item
# ---------------------------------------------------------

march_features = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) AS feature_impressions,
        SUM(gsc_clicks) AS feature_clicks,

        CASE
            WHEN SUM(gsc_impressions) > 0
            THEN 100.0 * SUM(gsc_clicks) / SUM(gsc_impressions)
        END AS feature_ctr,

        AVG(
            CASE
                WHEN gsc_impressions > 0
                THEN gsc_avg_position
            END
        ) AS feature_avg_position,

        STDDEV_SAMP(
            CASE
                WHEN gsc_impressions > 0
                THEN gsc_avg_position
            END
        ) AS feature_position_std,

        COUNT(
            DISTINCT CASE
                WHEN gsc_impressions > 0
                THEN report_date
            END
        ) AS feature_active_days

    FROM {FACT_MAR}

    GROUP BY
        client_hash_id,
        content_hash_id
""").df()


# Keep pages with enough March evidence
feature_df = march_features[
    march_features["feature_impressions"] >= 100
].copy()


# ---------------------------------------------------------
# Missing-value handling
# ---------------------------------------------------------

feature_df["feature_position_missing"] = (
    feature_df["feature_avg_position"].isna().astype(int)
)

position_median = feature_df[
    "feature_avg_position"
].median()

feature_df["feature_avg_position"] = (
    feature_df["feature_avg_position"]
    .fillna(position_median)
)

feature_df["feature_position_std"] = (
    feature_df["feature_position_std"]
    .fillna(0)
)


# ---------------------------------------------------------
# Engineer categorical position band
# ---------------------------------------------------------

def make_position_band(position):
    if position <= 3:
        return "top_3"
    elif position <= 10:
        return "page_1"
    elif position <= 20:
        return "striking"
    elif position <= 50:
        return "page_3_5"
    else:
        return "deep"


feature_df["feature_position_band"] = (
    feature_df["feature_avg_position"]
    .apply(make_position_band)
)


# One-hot encode categorical field
position_dummies = pd.get_dummies(
    feature_df["feature_position_band"],
    prefix="position",
    dtype=int
)


# ---------------------------------------------------------
# Final feature matrix
# ---------------------------------------------------------

numeric_features = [
    "feature_impressions",
    "feature_clicks",
    "feature_ctr",
    "feature_avg_position",
    "feature_position_std",
    "feature_active_days",
    "feature_position_missing",
]

X = pd.concat(
    [
        feature_df[numeric_features].reset_index(drop=True),
        position_dummies.reset_index(drop=True)
    ],
    axis=1
)


print(f"Feature rows: {len(X):,}")
print(f"Feature columns: {X.shape[1]}")
print(f"Missing values remaining: {X.isna().sum().sum()}")

assert X.isna().sum().sum() == 0

print("✓ Feature vector built successfully")

display(X.head())

## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

All model features come from the March 2026 feature window and therefore exist before the April outcome period.

- `feature_impressions`: total March GSC impressions. It measures search exposure.
- `feature_clicks`: total March GSC clicks.
- `feature_ctr`: March clicks divided by March impressions, multiplied by 100.
- `feature_avg_position`: average observed March search position. Missing values are filled with the median of the eligible March feature population.
- `feature_position_missing`: equals 1 when average position was originally missing. This preserves the fact that a value was absent instead of pretending the median was actually observed.
- `feature_position_std`: variation in observed March search position. Missing values, usually caused by insufficient repeated observations, are filled with 0 and interpreted carefully.
- `feature_active_days`: number of March days on which the page recorded at least one impression.
- `feature_position_band`: a categorical summary of average position (`top_3`, `page_1`, `striking`, `page_3_5`, or `deep`). It is one-hot encoded before modeling.

`client_hash_id` and `content_hash_id` remain context fields. They are needed to join feature and outcome data and later create grouped validation splits, but they are not model features.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

feature_notes = pd.DataFrame([
    {
        "feature": "feature_impressions",
        "meaning": "March search impressions",
        "missing_handling": "not missing after aggregation",
        "type": "numeric",
        "available_before_prediction": True
    },
    {
        "feature": "feature_clicks",
        "meaning": "March search clicks",
        "missing_handling": "not missing after aggregation",
        "type": "numeric",
        "available_before_prediction": True
    },
    {
        "feature": "feature_ctr",
        "meaning": "March click-through rate",
        "missing_handling": "population requires impressions >= 100",
        "type": "numeric",
        "available_before_prediction": True
    },
    {
        "feature": "feature_avg_position",
        "meaning": "Average March search position",
        "missing_handling": "median fill + missing indicator",
        "type": "numeric",
        "available_before_prediction": True
    },
    {
        "feature": "feature_position_std",
        "meaning": "Variation in March position",
        "missing_handling": "filled with 0",
        "type": "numeric",
        "available_before_prediction": True
    },
    {
        "feature": "feature_active_days",
        "meaning": "March days with impressions",
        "missing_handling": "count, no fill needed",
        "type": "numeric",
        "available_before_prediction": True
    },
    {
        "feature": "feature_position_missing",
        "meaning": "Position was originally missing",
        "missing_handling": "explicit indicator",
        "type": "binary",
        "available_before_prediction": True
    },
    {
        "feature": "feature_position_band",
        "meaning": "March average-position category",
        "missing_handling": "derived after position fill",
        "type": "categorical → one-hot",
        "available_before_prediction": True
    },
])

display(feature_notes)

assert feature_notes[
    "available_before_prediction"
].all()

print("✓ Every feature is available before April.")

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*


### 1. Future-window leakage

The feature window is March 1–31, 2026 and the outcome window is April 1–30, 2026. No April measurement is allowed in `X`.

### 2. Label-derived leakage

Fields used to construct the April opportunity proxy, including April impressions, clicks, CTR, average position, tier median CTR, CTR gap, and the final opportunity proxy, are forbidden from the feature matrix.

### 3. Product-decision leakage

Existing product scores, action flags, health scores, priority scores, or refresh decisions are not used. The model should learn from observable measurements rather than reproduce a rule already created by the product.

I also check for identifiers and privacy-sensitive fields. Pseudonymized client and content IDs remain outside the model features, and no raw client names, domains, URLs, titles, or private queries are used.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ---------------------------------------------------------
# Build April outcome only for leakage testing
# ---------------------------------------------------------

april_outcomes = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) AS outcome_impressions,
        SUM(gsc_clicks) AS outcome_clicks,

        CASE
            WHEN SUM(gsc_impressions) > 0
            THEN 100.0 * SUM(gsc_clicks) / SUM(gsc_impressions)
        END AS outcome_ctr,

        AVG(
            CASE
                WHEN gsc_impressions > 0
                THEN gsc_avg_position
            END
        ) AS outcome_avg_position

    FROM {FACT_APR}

    GROUP BY
        client_hash_id,
        content_hash_id
""").df()


# Join only to create/check the future proxy
model_frame = feature_df.merge(
    april_outcomes,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)


# ---------------------------------------------------------
# Create April-only outcome proxy
# ---------------------------------------------------------

eligible_outcome = model_frame[
    (model_frame["outcome_impressions"] >= 500)
    & model_frame["outcome_avg_position"].notna()
].copy()


def outcome_position_band(position):
    if position <= 3:
        return "top_3"
    elif position <= 10:
        return "page_1"
    elif position <= 20:
        return "striking"
    elif position <= 50:
        return "page_3_5"
    else:
        return "deep"


eligible_outcome["outcome_position_tier"] = (
    eligible_outcome["outcome_avg_position"]
    .apply(outcome_position_band)
)

eligible_outcome["outcome_tier_median_ctr"] = (
    eligible_outcome
    .groupby("outcome_position_tier")["outcome_ctr"]
    .transform("median")
)

eligible_outcome["ctr_gap_pp"] = (
    eligible_outcome["outcome_tier_median_ctr"]
    - eligible_outcome["outcome_ctr"]
)

eligible_outcome["opportunity_proxy"] = (
    eligible_outcome["ctr_gap_pp"] > 0.10
).astype(int)


# ---------------------------------------------------------
# Leakage checks
# ---------------------------------------------------------

feature_columns = set(X.columns)

future_terms = [
    "outcome",
    "april",
    "label",
    "proxy",
    "gap_pp",
]

future_leaks = [
    col for col in feature_columns
    if any(term in col.lower() for term in future_terms)
]


identifier_leaks = [
    col for col in feature_columns
    if col in {
        "client_hash_id",
        "content_hash_id",
        "url_hash_id",
        "keyword_hash_id",
    }
]


product_terms = [
    "health_score",
    "priority_score",
    "action_type",
    "refresh_flag",
    "needs_ctr_fix",
    "is_quick_win",
]

product_leaks = [
    col for col in feature_columns
    if any(term in col.lower() for term in product_terms)
]


privacy_terms = [
    "client_name",
    "domain",
    "url",
    "query",
    "keyword_text",
    "title",
]

privacy_leaks = [
    col for col in feature_columns
    if any(term == col.lower() for term in privacy_terms)
]


print("Future/label-derived leaks:", future_leaks)
print("Identifier leaks:", identifier_leaks)
print("Product-decision leaks:", product_leaks)
print("Privacy-sensitive leaks:", privacy_leaks)

assert len(future_leaks) == 0
assert len(identifier_leaks) == 0
assert len(product_leaks) == 0
assert len(privacy_leaks) == 0


# ---------------------------------------------------------
# Timeline check
# ---------------------------------------------------------

FEATURE_END = pd.Timestamp("2026-03-31")
OUTCOME_START = pd.Timestamp("2026-04-01")

assert FEATURE_END < OUTCOME_START

print("\n✓ Timeline check passed:")
print("Features end:", FEATURE_END.date())
print("Outcome starts:", OUTCOME_START.date())

print("\n✓ No tested leakage or privacy fields found in X.")

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*


I deliberately excluded the following fields from the feature vector:

- `client_hash_id` — pseudonymized identifier used only for grouping and client-level validation.
- `content_hash_id` — pseudonymized identifier used only for joins and grain checks.
- `outcome_impressions` — measured during April, so it is future information.
- `outcome_clicks` — measured during April, so it is future information.
- `outcome_ctr` — measured during the April outcome window and contributes directly to the proxy.
- `outcome_avg_position` — April information unavailable at the March prediction point.
- `outcome_position_tier` — derived from future April position.
- `outcome_tier_median_ctr` — derived from April outcome data.
- `ctr_gap_pp` — directly contributes to the opportunity proxy and therefore would leak the answer.
- `opportunity_proxy` — this is the value to be predicted, never an input.
- Product health scores, priority scores, action flags, and refresh flags — these would encode an existing decision rather than independent observed evidence.
- Raw client names, domains, URLs, page titles, raw keywords, and private queries — excluded for privacy and because the pseudonymized release does not require them.

The model feature vector therefore contains only safe March measurements and engineered values derived from those March measurements.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
excluded_fields = {
    "client_hash_id": "context identifier; grouping/splitting only",
    "content_hash_id": "context identifier; joins only",

    "outcome_impressions": "future April measurement",
    "outcome_clicks": "future April measurement",
    "outcome_ctr": "future April measurement and proxy input",
    "outcome_avg_position": "future April measurement",
    "outcome_position_tier": "derived from future April data",
    "outcome_tier_median_ctr": "derived from future April data",
    "ctr_gap_pp": "direct input to the proxy definition",
    "opportunity_proxy": "target/proxy itself",

    "health_score": "existing product decision",
    "priority_score": "existing product decision",
    "action_type": "existing product decision",
    "needs_ctr_fix": "existing product decision",

    "client_name": "private/raw identity",
    "domain": "private/raw identity",
    "raw_url": "private/raw URL",
    "raw_query": "private search text",
    "content_title": "raw page text",
}

excluded_check = pd.DataFrame(
    [
        {"field": field, "reason": reason}
        for field, reason in excluded_fields.items()
    ]
)

display(excluded_check)

accidental_use = sorted(
    set(excluded_fields).intersection(set(X.columns))
)

print("Excluded fields accidentally present in X:", accidental_use)

assert accidental_use == []

print("✓ All excluded fields remain outside the feature matrix.")

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.